# Extended Data Fig. 3c (right) — endoderm markers with and without A83-01

Author: Tianlei He. Ported from `endoderm_markers_day2diff_vs_endoderm0-20260912-.ipynb`.

The panel plots *SOX17*, *GSC*, *CER1* and *FOXA2* CPM in cysts after two days of differentiation with
A83-01 (`day2diff`, n = 3) or without it (`day2 endoderm`, n = 2), one axis per gene, as bars
(mean ± s.d.) with replicates as points. CPM values are used exactly as computed by the sequencing
provider; nothing here re-normalizes them.

**Changes from the original notebook**
1. Input is the GEO series GSE347564 CPM table instead of the provider's per-order matrix. Column names
   change accordingly: `KN4J6R_10–12_cpm` → `day2diff_rep1–3`, `KN4J6R_13–14_cpm` →
   `day2_endoderm_rep1–2`. The original loaded a sixth sample, `KN4J6R_15`, which it labels "Endoderm
   protocol"; the panel does not plot it and it is not in the GEO series, so it is not loaded here.
2. Paths are relative to the repository, with a check that the input table exists; outputs go to `bulkseq/output/`.
3. The original drew the four genes in four cells that differ only in the gene name (its cells for
   GSC, FOXA2, SOX17 and CER1). They are one loop here.
4. Removed cells that the panel does not use: the pooled-CPM histogram and threshold table, the
   mean-CPM ≥ 2 filtered-table export (the plots read the unfiltered rows), earlier combined and
   per-gene plot variants including ones with Welch's t-tests (no test is reported for this panel),
   Excel exports, a marker heatmap, and three-group strip plots that include the "Endoderm protocol"
   sample.
5. Added the last cell, which writes the plotted values to `bulkseq/output/ed3c_plotted_values.tsv`.

In [ ]:
# %pip install pandas matplotlib numpy seaborn

from pathlib import Path
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(context="notebook", style="whitegrid")
plt.rcParams["figure.dpi"] = 120

REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "bulkseq" / "README.md").is_file())
CPM_TABLE = Path(os.environ.get(
    "BULK_CPM_TABLE", REPO / "data" / "GSE347564" / "GSE347564_bulk_RNAseq_cpm_all_samples.tsv.gz"))
OUT = REPO / "bulkseq" / "output"
OUT.mkdir(parents=True, exist_ok=True)

annotation_columns = ["gene_id", "gene_name", "gene_biotype"]

BAR_GROUPS = {
    "day2diff (beads)": [
        "day2diff_rep1",
        "day2diff_rep2",
        "day2diff_rep3",
    ],
    "day2 endoderm": [
        "day2_endoderm_rep1",
        "day2_endoderm_rep2",
    ],
}

cpm_cols = [c for cols in BAR_GROUPS.values() for c in cols]

PANEL_GENES = ["SOX17", "GSC", "CER1", "FOXA2"]

In [ ]:
if not CPM_TABLE.is_file():
    raise FileNotFoundError(f"CPM table not found: {CPM_TABLE}. See bulkseq/README.md.")
df = pd.read_csv(CPM_TABLE, sep="\t")

missing = [c for c in annotation_columns + cpm_cols if c not in df.columns]
if missing:
    raise ValueError(f"Matrix missing columns: {missing}")

plot_df = df[annotation_columns + cpm_cols].copy()
len(plot_df)

In [ ]:
plotted_rows = []

for sym in PANEL_GENES:
    plot_symbols = [sym]

    long_rows = []
    for sym in plot_symbols:
        row = plot_df[plot_df["gene_name"].str.upper() == sym].iloc[0]
        for gname, cols in BAR_GROUPS.items():
            for c in cols:
                v = float(row[c])
                long_rows.append(
                    {
                        "gene_name": sym,
                        "group": gname,
                        "cpm": v,
                    }
                )
                plotted_rows.append((row["gene_id"], sym, gname, c, v))

    long_df = pd.DataFrame(long_rows)

    fig, ax = plt.subplots(figsize=(6, 5.5))
    sns.barplot(
        data=long_df,
        x="gene_name",
        y="cpm",
        hue="group",
        order=plot_symbols,
        errorbar="sd",
        capsize=0.08,
        ax=ax,
        palette=["#1b9e77", "#7570b3"],
    )
    sns.stripplot(
        data=long_df,
        x="gene_name",
        y="cpm",
        hue="group",
        order=plot_symbols,
        dodge=True,
        jitter=0.15,
        ax=ax,
        palette=["#1b9e77", "#7570b3"],
        size=4,
        alpha=0.85,
        linewidth=0.5,
        edgecolor="black",
        legend=False,
        zorder=5,
    )

    ax.set_xlabel("")
    ax.set_ylabel("CPM")
    ax.set_title(f"{sym} — day2 diff vs day2 endoderm")
    ax.set_ylim(bottom=0)
    ax.legend(title="", bbox_to_anchor=(1.02, 1), loc="upper left")
    fig.tight_layout()
    pdf_path = OUT / f"{sym}_day2_compared_day2endo.pdf"

    fig.tight_layout()
    fig.savefig(pdf_path, format="pdf", bbox_inches="tight")
    plt.show()

    print(f"Saved: {pdf_path}")

In [ ]:
plotted = pd.DataFrame(
    [{"gene_id": row_gene_id, "gene_name": sym, "group": g, "sample": c, "cpm": v}
     for (row_gene_id, sym, g, c, v) in plotted_rows]
)
plotted.to_csv(OUT / "ed3c_plotted_values.tsv", sep="\t", index=False)
print(f"ED 3c: wrote {len(plotted)} plotted values to bulkseq/output/ed3c_plotted_values.tsv")